# Lab | Data Aggregation and Filtering

This notebook analyzes the **marketing_customer_analysis.csv** dataset from the bootcamp repo.

Dataset:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

We cover:
1. Filter customers with low claims (< 1000) and response == "Yes"
2. Segment analysis: average premium & CLV by policy type and gender (Yes responders) + compare claims
3. Customers per state (only states with > 500 customers)
4. Max, min, median CLV by education and gender
5. **Bonus:** Policies sold by state and month (pivot)
6. **Bonus:** Monthly policies for top 3 states
7. **Bonus:** Response rate by marketing channel (with melt)


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()


Shape: (10910, 26)


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


## 1) Filter: total_claim_amount < 1000 and response == 'Yes'

In [4]:
filtered_df = df[(df["Total Claim Amount"] < 1000) & (df["Response"] == "Yes")]
print("Filtered shape:", filtered_df.shape)
filtered_df.head()


Filtered shape: (1399, 26)


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


## 2) Yes-responders: average monthly premium & CLV by policy_type and gender + compare claims

In [7]:
responded_df = df[df["Response"] == "Yes"].copy()

segment_analysis = (
    responded_df
    .groupby(["Policy Type", "Gender"], dropna=False)
    .agg(
        avg_monthly_premium=("Monthly Premium Auto", "mean"),
        avg_clv=("Customer Lifetime Value", "mean"),
        avg_total_claim=("Total Claim Amount", "mean"),
        customers=("Customer", "count")
    )
    .round(2)
    .sort_values(["avg_clv", "avg_total_claim"], ascending=[False, True])
)

segment_analysis


avg_monthly_premium  avg_clv  avg_total_claim  \
Policy Type    Gender                                                  
Personal Auto  F                     99.00  8339.79           452.97   
Special Auto   M                     86.34  8247.09           429.53   
Corporate Auto M                     92.19  7944.47           408.58   
               F                     94.30  7712.63           433.74   
Special Auto   F                     92.31  7691.58           453.28   
Personal Auto  M                     91.09  7448.38           457.01   

                       customers  
Policy Type    Gender             
Personal Auto  F             540  
Special Auto   M              32  
Corporate Auto M             154  
               F             169  
Special Auto   F              35  
Personal Auto  M             536

### Notes for interpretation
- **Higher avg_clv** and **higher avg_monthly_premium** can indicate higher profitability.
- **Lower avg_total_claim** suggests lower risk.
- A segment with **high CLV / premium** and **low claims** tends to be the most attractive.


## 3) Customers per state (keep only states with > 500 customers)

In [8]:
state_counts = df["State"].value_counts()
states_over_500 = state_counts[state_counts > 500]

states_over_500


State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

## 4) Max, min, median CLV by education and gender

In [9]:
clv_stats = (
    df.groupby(["Education", "Gender"], dropna=False)["Customer Lifetime Value"]
      .agg(["max", "min", "median"])
      .round(2)
      .sort_index()
)

clv_stats


max      min   median
Education            Gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

### Conclusions (write-up)
- Compare **median** CLV across education & gender as a robust indicator (less sensitive to outliers).
- Use **max** as a sign of potential high-value customers, but it can be driven by rare outliers.


## Bonus 5) Policies sold by state and month (months as columns, states as rows)

In [10]:
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")
df["month"] = df["effective_to_date"].dt.month

policies_by_state_month = (
    df.pivot_table(index="state", columns="month", values="customer", aggfunc="count")
      .fillna(0)
      .astype(int)
)

policies_by_state_month


KeyError: 'effective_to_date'

## Bonus 6) Monthly policies for top 3 states by total policies

In [ ]:
top3_states = df["state"].value_counts().head(3).index.tolist()
top3_policies_by_month = policies_by_state_month.loc[top3_states]

top3_states, top3_policies_by_month


## Bonus 7) Response rate by marketing channel (using melt)

In [ ]:
# Response rate by sales_channel (simple)
sales_channel_response_rate = (
    df.groupby("sales_channel")["response"]
      .apply(lambda x: (x == "Yes").mean())
      .sort_values(ascending=False)
      .round(3)
)
sales_channel_response_rate


In [ ]:
# Unpivot two marketing-related columns and compute response rate per channel value
melted = pd.melt(
    df,
    id_vars=["response"],
    value_vars=["sales_channel", "renew_offer_type"],
    var_name="channel_type",
    value_name="channel"
)

channel_response_rate = (
    melted.groupby(["channel_type", "channel"])["response"]
          .apply(lambda x: (x == "Yes").mean())
          .sort_values(ascending=False)
          .round(3)
)

channel_response_rate
